# Room 2: intensity production

This notebook uses `room_2_simulation.json` beside this notebook to produce `raw.h5` → `photons.h5` → `histogram.h5`, then renders `intensity.png` with the shared codebase plotting functions.

Use the project Python environment as the kernel. For native simulation, launch Jupyter from the repository root after activating the runtime (Jupyter must be available):

```bash
source ./activate.sh --full
jupyter lab examples/example_room_2/intensity.ipynb
```

Plotting existing histogram data does not require the native runtime. Running the plotting cell overwrites `intensity.png`.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import h5py
import numpy as np
import matplotlib.pyplot as plt

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/synthetic_data_generation").is_dir()
             and (p / "examples/example_room_2/room_2_simulation.json").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from within the repository.")
sys.path.insert(0, str(root / "src"))
output_dir = root / "examples/example_room_2"
config_path = output_dir / "room_2_simulation.json"
histogram_path = output_dir / "histogram.h5"
config = json.loads(config_path.read_text())
stages = {stage["command"]: stage["options"] for stage in config["stages"]}
print(json.dumps(config, indent=2))


## Generate or reuse the histogram

From the repository root, the equivalent command is:

```bash
synthetic-data run --config examples/example_room_2/room_2_simulation.json
```

The pipeline traces the configured scene using its saved camera and seed, pixelizes photons, and forms a clean histogram using the configured window and bin count. It adds no sensor noise.

Existing data is reused by default. Set `regenerate = True` to rerun all stages and overwrite the simulation outputs. This requires the built native tracer and activated runtime. Both the depth and intensity notebooks use the same histogram, so generation is only needed once.


In [ ]:
regenerate = False
if regenerate or not histogram_path.is_file():
    subprocess.run(
        [sys.executable, "-m", "synthetic_data_generation", "run",
         "--config", str(config_path)],
        cwd=root, check=True,
    )
print(f"Using {histogram_path}")


## Load and check the histogram

Check the HDF5 dataset against the local configuration before plotting. If you change generation settings, enable regeneration to refresh the data; shape checks alone cannot detect every stale configuration.


In [ ]:
with h5py.File(histogram_path, "r") as handle:
    histogram = handle["stamped_histogram"][:]
expected_shape = (stages["pixelize"]["width"], stages["pixelize"]["height"],
                  stages["histogram"]["bin_number"])
assert histogram.shape == expected_shape, (histogram.shape, expected_shape)
assert np.isfinite(histogram).all() and (histogram >= 0).all()
counts = histogram.sum(axis=-1)
assert counts.sum() > 0, "The histogram contains no photons."
print(f"Histogram: {histogram.shape}; photons in window: {counts.sum():.0f}")


## Display and save intensity

Reuse `display_histogram_intensity_image()`, which sums the histogram bins for each pixel. Intensity is the photon count **inside the configured histogram window**, not all simulated photons or material reflectivity. Zero-count pixels are masked by the shared plotting function. No intensity clipping is applied.


In [ ]:
from spad_noise_injection.histogram import display_histogram_intensity_image

fig, ax = plt.subplots(figsize=(7, 6))
display_histogram_intensity_image(histogram, ax=ax)
ax.set(title="Room 2: histogram intensity", xlabel="Pixel column", ylabel="Pixel row")
fig.tight_layout()
fig.savefig(output_dir / "intensity.png", dpi=180)
plt.show()
plt.close(fig)
print(output_dir / "intensity.png")
